# Imports

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
  
if Path.cwd().name in ['ETL', 'data_science']:
    root = Path.cwd().parent
else:
    root = Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))
else:
    None

from utils import align_working_dir

align_working_dir('ETL')

# Configurations

In [ ]:
# Define relative paths
INPUT_FILE_EVENTS = "../data_temp/events_3_category.json"
OUTPUT_FOLDER = "../data_processed/"
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "events.json")

# Simple check to verify the file is where we think it is
if os.path.exists(INPUT_FILE_EVENTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_EVENTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_EVENTS}")

# Ensure the output directory exists
if os.path.exists(OUTPUT_FOLDER):
    print(f"✅ Setup complete. Output folder found: {OUTPUT_FOLDER}")
else:
    os.makedirs(OUTPUT_FOLDER)
    print(f"✅ Created Output folder: {OUTPUT_FOLDER}")

# Load raw data

In [ ]:
try:
    events_df = pd.read_json(INPUT_FILE_EVENTS)
    print(f"✅ Successfully loaded {len(events_df)} records.")
    
    print("\n📑 Available columns:", *events_df.columns, sep="\n")
    
except FileNotFoundError:
    print(f"❌ Error: The file {INPUT_FILE_EVENTS} was not found.")

# Check whether all items have value for `speaker`

In [ ]:
# Set pandas to display ALL rows instead of truncating
pd.set_option('display.max_rows', None)

events_without_speaker_df = events_df[events_df['speaker'].str.strip() == ""].copy()

print(f"✅ Found {len(events_without_speaker_df)} events with no speaker assigned.")

display(events_without_speaker_df[['id', 'title', 'speaker', 'category', 'date']])

# Filter cases to update speaker
[NOTE]  
Based on the category, we can infer the likely speaker for some events. For example, if an event is categorized as '*simontonösszehangolva*' or '*simontonkulb*', it's likely that the speaer is Zsuzsa.

In [ ]:
filter_term_1 ="simontonösszehangolva"
filter_term_2 = "jólneveldasárkányod"
add_speaker_1 = "Dr. Prezenszki Zsuzsanna"

In [ ]:
filtered_df_1 = events_without_speaker_df[
    events_without_speaker_df['category'].str.contains(filter_term_1, case=False, na=False)
].copy()

results_filter_term_1 = filtered_df_1[['id', 'title', 'date', 'category', 'speaker']]

print(f"✅ Found {len(results_filter_term_1)} matches.")

display((results_filter_term_1))

In [ ]:
# Update the 'speaker' column using the index of your filtered results
events_df.loc[filtered_df_1.index, 'speaker'] =add_speaker_1

print(f"✅ Updated {len(filtered_df_1)} rows with the new speaker.")

display(events_df.loc[filtered_df_1.index, ['id', 'title', 'date', 'category', 'speaker']])

In [ ]:
filtered_df_2 = events_without_speaker_df[
    events_without_speaker_df['category'].str.contains(filter_term_2, case=False, na=False)
].copy()

results_filter_term_2 = filtered_df_2[['id', 'title', 'date', 'category', 'speaker']]

print(f"✅ Found {len(results_filter_term_2)} matches.")

display((results_filter_term_2))

In [ ]:
# Update the 'speaker' column in the original DataFrame for the matched rows
events_df.loc[filtered_df_2.index, 'speaker'] = add_speaker_1

print(f"✅ Updated {len(filtered_df_2)} rows with the new speaker.")

display(events_df.loc[filtered_df_2.index, ['id', 'title', 'date', 'category', 'speaker']])

# Once again check the remaining events without speaker

In [ ]:
rest_events_without_speaker_df = events_without_speaker_df[
    (~events_without_speaker_df['category'].str.contains(filter_term_1, case=False, na=False)) &
    (~events_without_speaker_df['category'].str.contains(filter_term_2, case=False, na=False))
].copy()

print(f"✅ Found {len(rest_events_without_speaker_df)} remaining events without a speaker.")
display(rest_events_without_speaker_df[['id', 'title', 'date', 'speaker', 'category', 'url']])

# Apply "Egyéb" as a speaker for remaining records
[NOTE]  
The output of the previous cell shows, that in these cases there are more there one speaker, or the speaker is not really important, because it is e.g., a Christmas party.

In [ ]:
# Update the 'speaker' column for these specific rows in the main events_df
events_df.loc[rest_events_without_speaker_df.index, 'speaker'] = "Egyéb"

print(f"✅ Updated {len(rest_events_without_speaker_df)} events with speaker: 'Egyéb'")
display(events_df.loc[rest_events_without_speaker_df.index,['id', 'title', 'date', 'speaker', 'category']])

# Export to JSON

In [ ]:
# force_ascii=False is crucial for keeping Hungarian characters like ő, ú, é
if pd.api.types.is_datetime64_any_dtype(events_df['date']):
    events_df['date'] = events_df['date'].dt.strftime('%Y-%m-%d')

# date_format='iso' ensures dates are readable strings
# categories will be included as long as they are in the events_df
events_df.to_json(
    OUTPUT_FILE, 
    orient='records', 
    indent=4, 
    force_ascii=False, 
    date_format='iso'
)

print(f"✅ Success! Cleaned data is saved to: {OUTPUT_FILE}")